# SmartDs 2D XY Point Plot (Starter)

This notebook reads an already-2D BATSRUS slice file and plots the **raw data points** in the XY plane.

It intentionally reuses the library (`SmartDs`) and does not resample or reimplement file parsing.

In [ ]:
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm

from starwinds_analysis.smart_ds import SmartDs

plt.rcParams['figure.dpi'] = 120

## Choose a 2D File

Set `DATA_FILE` to a BATSRUS/Tecplot slice file that is already 2D (for example an `x=...` or `y=...` slice output).

In [ ]:
DATA_FILE = Path('/path/to/your/2d_slice_file.dat')  # change this
# Example (if you have local slice files):
# DATA_FILE = Path('/Users/dagfev/Documents/SWMFsoftware/BATSRUS-xuvneutrals/run_test/GM/IO2/y=0_var_1_n00002000.dat')

sds = SmartDs.from_file(str(DATA_FILE))
print('Title:', sds.title)
print('Zone :', sds.zone)
print('Points:', len(sds.points))
print('Variables:', len(sds.variables))

In [ ]:
# Inspect available fields
list(sds.variables)[:40]

## Pick XY Coordinates and a Field to Color By

For an XY-plane slice, these are commonly `X [R]` and `Y [R]`.

In [ ]:
X_FIELD = 'X [R]'
Y_FIELD = 'Y [R]'
COLOR_FIELD = 'Rho [amu/cm^3]'  # change as needed

x = np.asarray(sds.variable(X_FIELD), dtype=float)
y = np.asarray(sds.variable(Y_FIELD), dtype=float)
c = np.asarray(sds.variable(COLOR_FIELD), dtype=float)

mask = np.isfinite(x) & np.isfinite(y) & np.isfinite(c)
print('Finite plotted points:', int(mask.sum()), '/', x.size)

In [ ]:
# Raw-point XY plot (no resampling)
fig, ax = plt.subplots(figsize=(7, 6))

c_valid = c[mask]
use_log = c_valid.size > 0 and np.nanmin(c_valid) > 0
norm = LogNorm() if use_log else None

pts = ax.scatter(
    x[mask],
    y[mask],
    c=c_valid,
    s=4,
    cmap='viridis',
    norm=norm,
    linewidths=0,
    rasterized=True,
)

ax.set_aspect('equal')
ax.set_xlabel(X_FIELD)
ax.set_ylabel(Y_FIELD)
ax.set_title(COLOR_FIELD)
fig.colorbar(pts, ax=ax, label=COLOR_FIELD)
plt.show()

## Optional: Try a Derived Field via `SmartDs`

If your environment has `griblet` recipes configured, you can request derived fields on demand.

In [ ]:
# Uncomment if you want derived BATSRUS fields (and griblet is installed/configured):
# sds.add_batsrus_graph()
# mach_a = sds.variable('M_A [none]')
# np.nanmin(mach_a), np.nanmax(mach_a)